# PydanticAI 입문 가이드

이 노트북은 `analysis_agent.ipynb`, `automation_agent.ipynb`, `planner_agent.ipynb`에서 사용하는 **PydanticAI**의 핵심 개념을 단계별로 설명합니다.

## PydanticAI란?

PydanticAI는 LLM 기반 애플리케이션을 만들기 위한 Python 프레임워크입니다.

핵심 아이디어는 간단합니다:
1. **Agent** — LLM을 감싸는 객체. "누구에게 어떤 일을 시킬지" 정의
2. **Instructions** — Agent에게 주는 시스템 프롬프트 (역할, 규칙)
3. **Structured Output** — LLM 응답을 Pydantic 모델로 강제하여 JSON 형태로 받기
4. **Tools** — Agent가 호출할 수 있는 Python 함수 (검색, API 호출 등)

아래에서 하나씩 실습해보겠습니다.

## 1단계: 환경 설정

먼저 API 키를 로드하고 필요한 라이브러리를 임포트합니다.

- `dotenv`: `.env` 파일에 저장된 `OPENAI_API_KEY`를 환경변수로 로드
- `pydantic`: 구조화된 출력 스키마를 정의하는 데이터 검증 라이브러리
- `pydantic_ai`: LLM Agent를 만들고 실행하는 프레임워크

In [ ]:
# .env 파일에서 OPENAI_API_KEY 환경변수 로드
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel   # 구조화된 출력 스키마 정의용
from pydantic_ai import Agent    # LLM 기반 Agent 프레임워크

## 2단계: 가장 간단한 Agent

PydanticAI에서 Agent를 만드는 최소 코드는 단 2줄입니다.

```python
agent = Agent("openai:gpt-5.4")          # 1) Agent 생성 (사용할 LLM 지정)
result = await agent.run("안녕하세요!")    # 2) 실행
```

- `Agent(모델명)`: 어떤 LLM을 사용할지 지정합니다
- `await agent.run(프롬프트)`: Agent에게 메시지를 보내고 응답을 받습니다
- `result.output`: 응답 텍스트가 담겨 있습니다

> **주의:** Jupyter 노트북에서는 `await agent.run()`을 사용합니다.
> 일반 Python 스크립트에서는 `agent.run_sync()`를 사용하면 됩니다.

In [ ]:
# 가장 간단한 Agent: 모델만 지정하고 바로 실행
simple_agent = Agent("openai:gpt-5.4")

# Agent에게 메시지를 보내고 응답 받기
result = await simple_agent.run("PydanticAI를 한 줄로 설명해줘.")

# result.output에 LLM의 응답 텍스트가 담겨 있음
print(result.output)

## 3단계: Instructions (시스템 프롬프트)

`instructions` 파라미터로 Agent에게 **역할과 규칙**을 부여할 수 있습니다.
이는 ChatGPT의 "system prompt"와 같은 역할입니다.

```python
agent = Agent(
    "openai:gpt-5.4",
    instructions="너는 친절한 요리사야. 모든 답변을 요리 비유로 해줘."
)
```

`instructions`가 있으면 Agent는 매 요청마다 이 규칙을 따릅니다.

In [ ]:
# instructions로 Agent에게 역할 부여
chef_agent = Agent(
    "openai:gpt-5.4",
    instructions="너는 친절한 요리사야. 모든 답변을 요리 비유로 설명해줘. 한국어로 답변.",
)

# 같은 질문이라도 instructions에 따라 답변 스타일이 달라짐
result = await chef_agent.run("소프트웨어 테스트가 왜 중요한가요?")
print(result.output)

## 4단계: Structured Output (구조화된 출력)

기본 Agent는 자유 텍스트를 반환합니다. 하지만 실무에서는 **정해진 형식의 데이터**가 필요합니다.

`output_type`에 Pydantic 모델을 지정하면, LLM이 자유 텍스트 대신 **해당 스키마에 맞는 데이터**를 반환합니다.

### 왜 필요할까요?

| 자유 텍스트 | 구조화된 출력 |
|---|---|
| "이 영화는 8점입니다" | `{"title": "...", "score": 8, "genre": "SF"}` |
| 후속 처리 어려움 | DB 저장, API 연동 바로 가능 |

### 사용법

```python
class Movie(BaseModel):        # 1) 원하는 출력 형식을 Pydantic 모델로 정의
    title: str
    score: int

agent = Agent(
    "openai:gpt-5.4",
    output_type=Movie,          # 2) Agent에 output_type으로 지정
)
result = await agent.run(...)
movie = result.output           # 3) result.output이 Movie 타입으로 반환됨
print(movie.title)              #    → 속성으로 바로 접근 가능
```

> **실전 활용:** `analysis_agent.ipynb`에서 `CodeReview` 모델로 코드 리뷰 결과를, `automation_agent.ipynb`에서 `DailyReport` 모델로 일일 보고서를 구조화합니다.

In [ ]:
# Step 1) 원하는 출력 형식을 Pydantic 모델로 정의
class MovieReview(BaseModel):
    title: str       # 영화 제목
    score: int       # 평점 (1~10)
    pros: list[str]  # 장점 목록
    cons: list[str]  # 단점 목록


# Step 2) output_type으로 지정 → LLM이 반드시 이 형식으로 응답
review_agent = Agent(
    "openai:gpt-5.4",
    output_type=MovieReview,
    instructions="영화 평론가로서 요청받은 영화를 평가. 한국어로 작성.",
)

# Step 3) 실행 → result.output이 MovieReview 타입
result = await review_agent.run("인터스텔라를 평가해주세요.")
review = result.output

# Pydantic 모델이므로 속성으로 바로 접근 가능
print(f"제목: {review.title}")
print(f"평점: {review.score}/10")
print(f"장점: {', '.join(review.pros)}")
print(f"단점: {', '.join(review.cons)}")

## 5단계: Tools (도구)

Agent에게 **Python 함수를 도구로 등록**하면, LLM이 필요할 때 해당 함수를 호출할 수 있습니다.

### 왜 필요할까요?

LLM은 혼자서는 다음을 할 수 없습니다:
- 실시간 데이터 조회 (날씨, 주가, DB)
- 외부 시스템 조작 (이메일 발송, 파일 저장)
- 계산이나 코드 실행

Tool을 등록하면 LLM이 **"이 함수를 호출해야겠다"고 판단**하고, 프레임워크가 실제 함수를 실행합니다.

### 사용법

```python
@agent.tool_plain          # 데코레이터로 도구 등록
def add(a: int, b: int) -> str:
    """두 숫자를 더한다."""     # docstring이 LLM에게 도구 설명으로 전달됨
    return str(a + b)
```

- `@agent.tool_plain`: Agent 컨텍스트 없이 독립적으로 동작하는 도구
- **함수 이름과 docstring**이 LLM에게 전달되어, LLM이 언제 이 도구를 쓸지 판단
- **타입 힌트** (`a: int, b: int`)가 LLM에게 파라미터 형식을 알려줌

> **실전 활용:** `automation_agent.ipynb`에서 `fetch_github_issues`, `send_slack_message`를, `planner_agent.ipynb`에서 `search_web`, `write_section`을 도구로 등록합니다.

In [ ]:
# Tool이 있는 Agent 만들기: 간단한 계산기 예제
calc_agent = Agent(
    "openai:gpt-5.4",
    instructions="계산이 필요하면 반드시 제공된 도구를 사용하라. 한국어로 답변.",
)


# @agent.tool_plain 데코레이터로 도구 등록
# - 함수 이름(add)과 docstring이 LLM에게 전달됨
# - LLM이 "덧셈이 필요하다"고 판단하면 이 함수를 호출
@calc_agent.tool_plain
def add(a: int, b: int) -> int:
    """두 숫자를 더한다."""
    print(f"  [Tool 호출] add({a}, {b})")
    return a + b


@calc_agent.tool_plain
def multiply(a: int, b: int) -> int:
    """두 숫자를 곱한다."""
    print(f"  [Tool 호출] multiply({a}, {b})")
    return a * b


# Agent 실행 — LLM이 알아서 필요한 도구를 선택하여 호출
result = await calc_agent.run("17과 28을 더하고, 그 결과에 3을 곱해주세요.")
print(f"\n최종 답변: {result.output}")

## 6단계: 모두 합치기 — Tool + Structured Output

지금까지 배운 것을 조합하면 **실전에서 쓸 수 있는 Agent**가 됩니다.

이 예제에서는 도시 이름을 받아 날씨를 조회(Tool)하고, 구조화된 여행 추천(Structured Output)을 반환하는 Agent를 만듭니다.

```
사용자: "서울 여행 추천해줘"
   ↓
Agent가 get_weather("서울") Tool 호출
   ↓
LLM이 날씨 정보를 참고해 TravelAdvice 형식으로 응답
   ↓
result.output → TravelAdvice(city="서울", weather="맑음", ...)
```

In [ ]:
# --- 구조화된 출력 정의 ---
class TravelAdvice(BaseModel):
    city: str              # 도시 이름
    weather: str           # 현재 날씨
    recommended: bool      # 여행 추천 여부
    tips: list[str]        # 여행 팁 목록


# --- Agent 생성: Tool + Structured Output 조합 ---
travel_agent = Agent(
    "openai:gpt-5.4",
    output_type=TravelAdvice,  # 구조화된 형식으로 응답 강제
    instructions=(
        "여행 가이드로서 도시의 날씨를 조회하고 여행 조언을 제공한다. "
        "반드시 get_weather 도구로 날씨를 확인한 후 답변하라. 한국어로 작성."
    ),
)


# --- Tool 등록: 날씨 조회 (Mock) ---
@travel_agent.tool_plain
def get_weather(city: str) -> str:
    """도시의 현재 날씨 정보를 조회한다."""
    # 실제로는 날씨 API를 호출하지만, 여기서는 Mock 데이터 반환
    mock_data = {
        "서울": "맑음, 22°C, 습도 45%",
        "부산": "흐림, 19°C, 오후 비 예보",
        "제주": "맑음, 24°C, 바람 강함",
    }
    weather = mock_data.get(city, f"{city}: 데이터 없음, 약 20°C 예상")
    print(f"  [Tool 호출] get_weather('{city}') → {weather}")
    return weather


# --- 실행 ---
result = await travel_agent.run("제주도 여행을 계획 중이에요. 지금 가도 될까요?")
advice = result.output

print(f"도시: {advice.city}")
print(f"날씨: {advice.weather}")
print(f"추천: {'✅ 추천' if advice.recommended else '❌ 비추천'}")
print("팁:")
for tip in advice.tips:
    print(f"  - {tip}")

## 정리: PydanticAI 핵심 패턴 요약

| 개념 | 역할 | 코드 |
|------|------|------|
| **Agent** | LLM을 감싸는 핵심 객체 | `Agent("openai:gpt-5.4")` |
| **instructions** | 역할·규칙 부여 (시스템 프롬프트) | `Agent(..., instructions="...")` |
| **output_type** | 응답을 Pydantic 모델로 강제 | `Agent(..., output_type=MyModel)` |
| **@agent.tool_plain** | LLM이 호출할 수 있는 함수 등록 | `@agent.tool_plain` 데코레이터 |
| **await agent.run()** | Agent 실행 (Jupyter용) | `result = await agent.run("...")` |
| **result.output** | 응답 데이터 접근 | `result.output.title` |

### 다음 단계

이 개념들이 실전에서 어떻게 조합되는지 확인해보세요:

| 노트북 | Agent 유형 | 핵심 패턴 |
|--------|-----------|----------|
| `analysis_agent.ipynb` | 분석형 | output_type만 사용 (Tool 없음) |
| `automation_agent.ipynb` | 자동화형 | Tool + output_type (고정 순서) |
| `planner_agent.ipynb` | Planner형 | Tool만 사용 (LLM이 순서 결정) |